# HAKE-MER — Step 0 baseline (GoEmotions)

Definitive **flat PLM** campaign: **5 epochs max** (early stopping patience 1), seeds 42 / 123 / 456, metrics F1-micro, F1-macro, exact match, mAP.

Runs **DistilBERT-base** then **RoBERTa-base** via `run_baseline_campaign.sh`.

### Run

1. **Runtime → Change runtime type → GPU** (T4 is enough).
2. **Runtime → Run all** (expect **~2–4 hours** for both backbones on a T4).
3. Download the zip from the last cell and send it back for chapter 4 update.

Use **Runtime → Factory reset runtime** before a fresh rerun so old checkpoints do not mix with new ones.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Use Runtime → Change runtime type → GPU, then run this cell again."
    )
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO = "khalef-khalil/marii"
WORKDIR = Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"


def git_token() -> str:
    try:
        from google.colab import userdata

        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return getpass("GitHub token with repo read access (input hidden): ")


def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=True)
        return

    r = subprocess.run(
        ["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)],
        capture_output=True,
    )
    if r.returncode == 0 and (WORKDIR / "run_baseline_campaign.sh").is_file():
        return

    token = git_token()
    if WORKDIR.is_dir():
        subprocess.run(["rm", "-rf", str(WORKDIR)], check=True)
    authed = f"https://{token}@github.com/{REPO}.git"
    subprocess.run(["git", "clone", "--depth", "1", authed, str(WORKDIR)], check=True)


clone_repo()
%cd {WORKDIR}
print("Commit:", end=" ")
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
!./run_baseline_campaign.sh --backbone distilbert-base-uncased --epochs 5

In [ ]:
!./run_baseline_campaign.sh --backbone roberta-base --epochs 5

In [ ]:
import json
from pathlib import Path

artifact_dir = Path("reference/artifacts")
campaign_files = sorted(artifact_dir.glob("baseline_plm_*_campaign.json"))
if not campaign_files:
    raise FileNotFoundError("No campaign JSON found. Run both training cells first.")

for path in campaign_files:
    campaign = json.loads(path.read_text(encoding="utf-8"))
    print(f"\n=== {path.name} ({campaign.get('backbone', '?')}) ===")
    for name, block in campaign["test_aggregate"].items():
        print(f"  {name}: {block['mean']:.4f} ± {block['std']:.4f}")

In [ ]:
import zipfile
from google.colab import files

zip_path = Path("/content/baseline_plm_step0_5epoch_results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for campaign in sorted(Path("reference/artifacts").glob("baseline_plm_*_campaign.json")):
        zf.write(campaign, campaign.name)
    for metrics_file in sorted(Path("runs").glob("*_baseline_plm/metrics.json")):
        arcname = f"{metrics_file.parent.name}/{metrics_file.name}"
        zf.write(metrics_file, arcname)

print(f"Zip size: {zip_path.stat().st_size / 1e6:.1f} MB (campaign summaries + per-seed metrics)")
files.download(str(zip_path))